# **ColabSeqDisplay: fit a model to your own variant library, in a browser.**

<img src="https://img.shields.io/badge/Paper-not%20yet%20posted-lightgrey" style="max-width: 100%;">
<a href="https://github.com/JasonJiangs/ColabSeqDisplay"><img src="https://img.shields.io/badge/Github-black?logo=github" style="max-width: 100%;"></a>
<a href="https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay.ipynb"><img src="https://img.shields.io/badge/Open%20in-Colab-F9AB00?logo=googlecolab&logoColor=white" style="max-width: 100%;"></a>
<a href="https://github.com/JasonJiangs/ColabSeqDisplay"><img src="https://img.shields.io/badge/License-see%20repository-lightgrey" style="max-width: 100%;"></a>

- You measured a **combinatorial variant library** — one row per variant, one column per mutated site, one column per assay condition. This notebook fine-tunes a protein language model on it with LoRA and hands you something that scores variants you have not made yet.

- **For the person who ran the assay.** No code, no YAML, no conda: every choice is a field in the panel below, and the modelling hyperparameters are looked up for the backbone you pick. What you do set is the budget: how many epochs at most, when to give up, and how many sequences sit on the GPU at once.

- **Try it with nothing of your own.** The first field offers a bundled example — MG8 PETases, 124 variants of a 287-residue wild type across 19 mutated sites in single-letter code, with one measured condition (`activity`) — small enough to run end to end inside one free Colab session, and far too small to measure a backbone with.

- **Two files leave here**: `model_bundle.zip`, the trained model, and `performance_report.zip`, what it scored. No hub, no account, no upload — nothing you load leaves this runtime.

- **Two notebooks.** [ColabSeqDisplay](https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay.ipynb) trains a model on your library and exports it. [ColabSeqDisplay_Predict](https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay_Predict.ipynb) scores new variants from a saved `.zip`.

- **The science is not ours.** LoRA injection, the training loop, the read-out, metrics and splits come from the *SequenceDisplay-Workflow-Optimization* research package (`seqdisplay_opt`), vendored into `colabsd/engine/`; `ATTRIBUTION.md` names every file it came from.

# How to start

## 1 · Switch this runtime to a GPU

`Runtime` ▸ `Change runtime type` ▸ **T4 GPU** ▸ `Save`. Colab restarts the runtime and clears anything you had already run.

## 2 · Click the run-button

Hover over the cell below and click ▶ on its left. It installs the package (1–2 minutes the first time), then draws the panel that **is** this notebook. There is no code to write and no other cell to edit.

**Two buttons, not one.** The panel reports validation numbers only, including the performance archive it exports. Reading the locked test partition is the last cell — a separate, counted, deliberate act, explained there.

## 3 · Which GPU

- **T4 — <font color="red">free</font>.** 16 GB, enough for most of this. <font color="red">Free sessions are pre-empted and capped in length, so a run measured in hours is a run you will probably lose. Mount Drive first — see below.</font>
- **L4 (needs Colab Pro).** 24 GB — the smallest card that runs the one backbone that will not fit a T4 (`SaProt-1.3B`), and a steadier session.
- **A100 (needs Colab Pro).** 40 GB and much faster; the card for a full multi-seed evaluation.
- **No GPU (CPU runtime).** The panel opens, and loads and checks your library. Anything that needs the language model itself — training, or folding a structure — refuses to start and says which runtime it needs.

### The free tier will disconnect. Mount Drive before it does.

`Files` — the folder icon in the left margin — then **Mount Drive**. Everything outside `/content/drive/MyDrive/` is thrown away when the session ends, including the backbone download and anything the panel wrote, so copy what you want to keep into Drive as it appears — `model_bundle.zip` and `performance_report.zip` above all, and a wild-type 3Di string you folded here, which is the slowest thing on this page to make twice.

### Which backbone fits which card

| backbone | embedding width | needs a 3Di string | min/run · study library (16,424) | min/run · bundled example (124) | where to run it |
|---|---|---|---|---|---|
| `ESM2-8M` | 320-d | no | 20 | 1 min | a free T4 |
| `ESM2-35M` | 480-d | no | 40 | 1 min | a free T4 |
| `ESM2-150M` | 640-d | no | 90 | 1 min | a free T4, if you mount Drive first |
| `ESM2-650M` | 1280-d | no | 240 | 2 min | **L4 or A100** — a free session usually drops first |
| `SaProt-35M` | 480-d | **yes** | 45 | 1 min | a free T4 |
| `SaProt-650M` | 1280-d | **yes** | 260 | 2 min | **L4 or A100** — a free session usually drops first |
| `SaProt-1.3B` | 1280-d | **yes** | — | — | **L4 or A100** — will not fit a T4 |

**Neither minutes column is a measurement.** The first is the registry's order-of-magnitude figure for one training run on a T4 over the library the hyperparameter study used — 16,424 variants of a 1,054-residue protein, not in this repository. The second is that figure rescaled by row count to the bundled example (124 variants of 287 residues), which over-states it; it never estimates below a minute, so `1 min` means *under* one. The panel prints the real rate once it is training — trust that. Multiply either column by 9 for a full evaluation: 3 data splits x 3 model seeds.

**Where to run it** reads the first minutes column: ≤ 60 min per run is comfortable on a free T4, up to 150 min if your results are on Drive, beyond that a free session is likely to end first. A blank estimate means the model does not fit a T4's 16 GB at all, however small your library is.

**The modelling hyperparameters are looked up, not tuned by you.** They are read from `config/best/` for the backbone you pick, and none of the seven on this form carries tuned values — every one is a **placeholder**, so the panel raises a **Warning** the moment you pick one and every number it prints is a lower bound rather than a result.

**The budget is yours.** Three boxes, prefilled from that entry and yours from then on: *Epochs, at most*, *Give up after this many epochs with no gain*, *Sequences on the GPU at once (memory)* — the last is the one to lower when the GPU runs out of memory. `adapter_lr`, `head_lr`, `adapter_weight_decay`, `lora_rank`, `lora_alpha`, `lora_dropout`, `effective_batch_size` stay read-only. What you change travels into both files you leave with, marked as yours.

A run is one data split x one model seed. `config/best/` calls 3 x 3 = 9 runs a full evaluation; the panel defaults to **one** run — enough to see the machinery works, not enough to quote a ±.

### How long this takes

**The bundled example is minutes.** Every backbone the table gives an estimate for — the default `ESM2-35M` included — is 2 min or less for one run over its 124 variants; the install and the one-off backbone download are most of the wait.

**Your own library scales with rows.** The panel re-does the arithmetic once it has read your table and prints it above the Train button. At the study library's 16,424 variants the same `ESM2-35M` is 40 min for one run, and a full 3 x 3 evaluation of `SaProt-650M` — the slowest backbone the table puts a number on — is 39 h 00 min.

<font color="red">**One free session is worth about two hours**</font>, and Colab can reclaim it sooner. The panel warns you before you press Train when its own estimate does not fit; mount Drive first so a drop costs you the run rather than the afternoon.

### Where the other backbones went

The form offers seven backbones — the ESM2 ladder and SaProt — so that step 2 stays one real comparison, sequence against sequence-and-shape, rather than a menu of fourteen. The other seven are in the package but not on the form:

- `ProtT5-XL` — a 1.2B-parameter encoder that needs an L4 or A100 and an extra sentencepiece install, where the notebooks target a free T4 and install nothing beyond colabsd.
- `Ankh-large` — a 1.2B-parameter encoder that needs an L4 or A100, where the notebooks target a free T4.
- `ESMC-300M` — it loads only through the EvolutionaryScale SDK (`pip install esm`), an install the notebooks do not make on a user's behalf.
- `ESMC-600M` — it loads only through the EvolutionaryScale SDK (`pip install esm`), an install the notebooks do not make on a user's behalf.
- `SeqDance` — a dynamics-pretrained ESM2-35M — a good model, but it answers a narrower question than the sequence-versus-structure choice the notebooks are built around.
- `ESMDance` — a dynamics-tuned ESM2-35M whose pooled feature is its 50-dim prediction head rather than the trunk, so it is not read like the other entries in a single comparison.
- `METL` — Rosetta-pretrained and protein-specific, with no HuggingFace weights for anything to load.

All of these keep their `config/best/` entry, and all but `METL` keep a tested adapter that `create_adapter()` still builds. A `model_bundle.zip` trained on one of them still scores in the Predict notebook. To put a family back on the form, move its name from `WITHHELD_FAMILY_REASONS` to `OFFERED_FAMILIES` in `colabsd/backbones/registry.py`.

### The 3Di step appears only for a SaProt backbone

A **SaProt** backbone reads your protein's shape as well as its sequence, so it needs a wild-type **3Di string**, and the step that produces one appears under the backbone choice. Pick an **ESM2** and that step is gone and the later steps renumber. The string stays in this session — written into the work folder as `wt_3di.txt`, offered back rather than recomputed for the same wild type, and never downloaded and uploaded again.

### What that step costs

**Almost nothing — if you have a structure.** Download your protein's model from [alphafold.ebi.ac.uk](https://alphafold.ebi.ac.uk), upload the `.cif`, and the conversion takes seconds on any runtime. Nothing in this repository ships a 3Di string, the bundled example included, so every **SaProt** run starts with a structure of your own wild type. With an **ESM2** there is nothing to prepare at all.

**Folding it yourself is the last resort.** <font color="red">ESMFold memory grows with the square of the length: `colabsd.structure` calls **700 residues** the most a free 16 GB T4 can be expected to fold, warns you past it, and will not fold at all until you tick the memory-risk box.</font> Try AlphaFold first — faster, free and more accurate.

### The two files you leave with

- **`model_bundle.zip`** — the model: LoRA weights, head, your library description, the *frozen* coordinates of your mutated sites, the hyperparameters, the budget the run was actually given — marking anything you set yourself — and the provenance. This is what the Predict notebook asks for, and it needs no 3Di string — the one this run used is inside it.
- **`performance_report.zip`** — the numbers: `report.csv`, `report.png`, `report.json`, plus a `performance.json` and a `README.txt` stating which partition they describe and how many times the test set has been read.

`performance_report.zip` needs no unlock: press its button with the test partition still locked and it comes back full of **validation** numbers and says so. Unlocking in the last cell rewrites the same archive with the test numbers and the unlock count.

In [ ]:
#@title **Click the run-button to use ColabSeqDisplay** { display-mode: "form" }

#@markdown ### Hint
#@markdown - **A file picker freezes this page while it is open.** Every control stops responding until you pick a file or press **Cancel upload**. The panel names the file it is waiting for before the dialog opens.
#@markdown - **The ▶ button.** It spins while the cell installs the package and builds the panel, then goes back to ▶ — that means finished, not broken. The panel stays live after the cell ends; if it stops responding, click ▶ again to rebuild it.
#@markdown ### <font color=red>If the session disconnects</font>
#@markdown - <font color=red>Reconnect and run this cell again: the panel comes back, but the runtime is empty. Whatever was under `/content` is gone; whatever you wrote to a mounted Google Drive folder is not. Mount Drive before you start anything long.</font>
#@markdown - <font color=red>Changing the runtime type empties it the same way. Stop this cell first, change the runtime, then run it again.</font>
#@markdown ### Where the code comes from
#@markdown - The field below names what gets installed — one repository, fine-tuning engine included. A **folder path** works as well as a URL, and is installed with `pip install -e`.
colabsd_repository = "https://github.com/JasonJiangs/ColabSeqDisplay.git"  #@param {type:"string"}

import importlib
import importlib.util
import subprocess
import sys
from pathlib import Path

WORK_ROOT = Path.cwd()


def run_command(command):
    """Run a command, raising with its own output when it fails."""
    parts = [str(part) for part in command]
    finished = subprocess.run(parts, capture_output=True, text=True)
    if finished.returncode != 0:
        raise RuntimeError(
            "This command failed:\n  " + " ".join(parts) + "\n"
            + (finished.stdout or "")[-1500:] + (finished.stderr or "")[-1500:]
        )
    return finished


def head_of(repo):
    """The commit a checkout is on, or "" when it is not a git repository."""
    try:
        return run_command(["git", "-C", str(repo), "rev-parse", "HEAD"]).strip()
    except RuntimeError:
        return ""


def checkout(source, name):
    """A local folder as given, or a clone of a git URL beside this notebook, brought up to date.

    An existing clone is fetched and reset onto the remote rather than left alone. Colab keeps a
    runtime alive across many hours: without this, a fix published after the first run of the
    session can never reach the user, because the package is already installed and the old
    fast path did nothing at all.
    """
    local = Path(source).expanduser()
    if local.is_dir():
        return local.resolve(), False
    target = WORK_ROOT / name
    if not (target / ".git").is_dir():
        print("cloning " + str(source) + " ...")
        try:
            run_command(["git", "clone", "--depth", "1", source, target])
        except RuntimeError as exc:
            raise RuntimeError(
                str(exc) + "\n\n" + name + " could not be downloaded from " + str(source) + ". In the "
                "field at the top of this form, put a repository this runtime can reach, or the path of a "
                "folder you uploaded to it (for example " + str(target) + ")."
            ) from None
        return target.resolve(), True

    before = head_of(target)
    try:
        run_command(["git", "-C", str(target), "fetch", "--depth", "1", "origin"])
        run_command(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"])
    except RuntimeError:
        print("could not check " + str(source) + " for updates; using the copy already here")
        return target.resolve(), False
    moved = head_of(target) != before
    if moved:
        print("updated " + name + " to " + head_of(target)[:7])
    return target.resolve(), moved


def package_dir(module):
    """The directory an importable package sits in, or None when it is not importable."""
    found = importlib.util.find_spec(module)
    if found is None or not found.origin:
        return None
    return Path(found.origin).resolve().parent


def colabsd_is_complete():
    """True when colabsd is importable *and* its config registry and bundled example came with it.

    Two layouts are both correct: an editable install leaves `config/` and `examples/` beside the
    package, a built wheel carries them inside it. Either answer counts; neither does.
    """
    package = package_dir("colabsd")
    if package is None:
        return False
    return any(
        (root / "config" / "best").is_dir() and (root / "examples").is_dir()
        for root in (package, package.parent)
    )


package_root, moved = checkout(colabsd_repository, "ColabSeqDisplay")
if moved or not colabsd_is_complete():
    print("installing ColabSeqDisplay from " + str(package_root) + " ...")
    run_command([sys.executable, "-m", "pip", "install", "-q", "-e", package_root])
    importlib.invalidate_caches()
    if str(package_root) not in sys.path:
        sys.path.insert(0, str(package_root))
    for module in [name for name in sys.modules if name == "colabsd" or name.startswith("colabsd.")]:
        del sys.modules[module]

import colabsd
from colabsd.ui import core, main_workflow

runtime = core.detect_runtime()
WORK_DIR = WORK_ROOT / "colabsd_work"

if runtime.has_gpu:
    GPU_DESCRIPTION = str(runtime.gpu_name) + "  (" + format(runtime.gpu_memory_gb or 0.0, ".1f") + " GB)"
else:
    GPU_DESCRIPTION = "none — Runtime > Change runtime type > T4 GPU, then run this cell again"

print("colabsd " + colabsd.__version__ + "   from " + str(Path(colabsd.REPO_ROOT)))
print("GPU       " + GPU_DESCRIPTION)
print("files     " + str(WORK_DIR))
print("")

wizard = main_workflow.launch(work_dir=WORK_DIR)

In [ ]:
#@title **Click the run-button to unlock the test set and write the final report** { display-mode: "form" }

#@markdown ### Why this is a separate button
#@markdown - Everything the panel above reports is measured on **validation**, on purpose. You chose the backbone and the number of runs by looking at those numbers, so they are no longer an honest estimate of how the model behaves on data nobody has looked at.
#@markdown - The **test** partition stays locked through all of that and is read here: once, deliberately, by you. A number you consult while you are still deciding stops being a test number.
#@markdown - **Every unlock is counted** — written to `unlock.json` beside the run and printed in the report, so whoever reads the result can see how many times the test set was opened. Unlock once, at the end. Train again afterwards and you unlock again; the count goes up.
#@markdown ### What this cell writes
#@markdown - It rewrites `performance_report.zip` — the report table, the figure and the JSON — so that it now carries the **test** numbers and the unlock count, and offers it back along with the figure. The panel above already wrote that archive with the **validation** numbers, which is what to read while you are still deciding anything.

try:
    from colabsd.ui import main_workflow, unlock
except ImportError:
    raise RuntimeError(
        "ColabSeqDisplay is not installed in this runtime yet. Run the cell above first."
    ) from None

try:
    trained = wizard
except NameError:
    raise RuntimeError(
        "Run the cell above first and train something in the panel it draws. This cell reports on that "
        "run, which it reads from the `wizard` that cell leaves behind."
    ) from None

unlock.launch(
    trained,
    output_dir=main_workflow.run_dir(trained.state),
    work_dir=main_workflow.work_dir(trained.state),
)